# 06 - Share of voice va competitor benchmark

Notebook tinh SOV theo reach/engagement/post count tu `vw_competitor_benchmark.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

competitors = read_csv('dashboard/exports/vw_competitor_benchmark.csv')
brand_name = 'Highlands Coffee Vietnam'


In [ ]:
total_reach = competitors['total_reach'].sum()
total_engagement = competitors['total_engagement'].sum()
benchmark = competitors.assign(
    reach_sov=np.where(total_reach > 0, competitors['total_reach'] / total_reach * 100, np.nan),
    engagement_sov=np.where(total_engagement > 0, competitors['total_engagement'] / total_engagement * 100, np.nan),
    post_sov=competitors['post_count'] / competitors['post_count'].sum() * 100,
).sort_values('reach_sov', ascending=False)
display(benchmark[['page_name', 'is_competitor', 'post_count', 'total_reach', 'total_engagement', 'avg_engagement_rate', 'reach_sov', 'engagement_sov', 'post_sov']].round(4))


In [ ]:
brand_vs_competitor_flag = competitors.groupby('is_competitor', as_index=False).agg(
    post_count=('post_count', 'sum'),
    total_reach=('total_reach', 'sum'),
    total_engagement=('total_engagement', 'sum'),
    avg_engagement_rate=('avg_engagement_rate', 'mean'),
)
brand_vs_competitor_flag['reach_sov'] = brand_vs_competitor_flag['total_reach'] / brand_vs_competitor_flag['total_reach'].sum() * 100
brand_vs_competitor_flag['engagement_sov'] = brand_vs_competitor_flag['total_engagement'] / brand_vs_competitor_flag['total_engagement'].sum() * 100
display(brand_vs_competitor_flag.round(4))


In [ ]:
brand_row = benchmark[benchmark['page_name'].eq(brand_name)]
if brand_row.empty:
    raise ValueError(f'Main brand not found: {brand_name}')
display(brand_row[['page_name', 'post_count', 'total_reach', 'total_engagement', 'avg_engagement_rate', 'reach_sov', 'engagement_sov']].round(4))
display(benchmark.head(5)[['page_name', 'total_reach', 'reach_sov', 'total_engagement', 'engagement_sov']].round(4))


## Insight

- Key finding: Highlands dan SOV theo reach trong tap du lieu hien tai, nhung engagement SOV van thap hon muc reach SOV.
- Supporting data: Highlands co 99.40% reach SOV, 82.02% engagement SOV va 25 posts; competitor flag chiem 0.20% reach SOV.
- Recommended action: Khong chi toi uu so luong post; can tang noi dung tao phan hoi/binh luan de cai thien engagement SOV.